In [ ]:
# In[1]:

import pandas as pd
import pytz

# load CSVs (they are in the current working directory)
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")

# timezone for UTC+8
tz = pytz.timezone("Asia/Shanghai")

# compute overall timestamp ranges and present both epoch and localized datetime (compact)
metric_min_ts = int(df_metric['timestamp'].min()) if not df_metric.empty else None
metric_max_ts = int(df_metric['timestamp'].max()) if not df_metric.empty else None
trace_min_ts = int(df_trace['timestamp'].min()) if not df_trace.empty else None
trace_max_ts = int(df_trace['timestamp'].max()) if not df_trace.empty else None

metric_range = pd.Series({
    'min_timestamp_epoch': metric_min_ts,
    'max_timestamp_epoch': metric_max_ts,
    'min_timestamp_local': pd.to_datetime(metric_min_ts, unit='s', utc=True).tz_convert(tz) if metric_min_ts is not None else None,
    'max_timestamp_local': pd.to_datetime(metric_max_ts, unit='s', utc=True).tz_convert(tz) if metric_max_ts is not None else None
})

trace_range = pd.Series({
    'min_timestamp_epoch': trace_min_ts,
    'max_timestamp_epoch': trace_max_ts,
    'min_timestamp_local': pd.to_datetime(trace_min_ts, unit='s', utc=True).tz_convert(tz) if trace_min_ts is not None else None,
    'max_timestamp_local': pd.to_datetime(trace_max_ts, unit='s', utc=True).tz_convert(tz) if trace_max_ts is not None else None
})

# Aggregations for metric.csv: group by cmdb_id and kpi_name
if not df_metric.empty:
    metric_grouped = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg([
            ('count', 'count'),
            ('p95', lambda s: s.quantile(0.95)),
            ('p90', lambda s: s.quantile(0.90)),
            ('p50', 'median'),
            ('p5', lambda s: s.quantile(0.05))
        ])
        .reset_index()
    )
    # round numeric columns for compactness
    metric_grouped[['p95','p90','p50','p5']] = metric_grouped[['p95','p90','p50','p5']].round(6)
    metric_summary = metric_grouped.sort_values('count', ascending=False).head(50)
else:
    metric_summary = pd.DataFrame(columns=['cmdb_id','kpi_name','count','p95','p90','p50','p5'])

# Aggregations for trace.csv: group by cmdb_id and trace_name
if not df_trace.empty:
    trace_grouped = (
        df_trace
        .groupby(['cmdb_id', 'trace_name'])['value']
        .agg([
            ('count', 'count'),
            ('p95', lambda s: s.quantile(0.95)),
            ('p90', lambda s: s.quantile(0.90)),
            ('p50', 'median'),
            ('p5', lambda s: s.quantile(0.05))
        ])
        .reset_index()
    )
    trace_grouped[['p95','p90','p50','p5']] = trace_grouped[['p95','p90','p50','p5']].round(6)
    trace_summary = trace_grouped.sort_values('count', ascending=False).head(50)
else:
    trace_summary = pd.DataFrame(columns=['cmdb_id','trace_name','count','p95','p90','p50','p5'])

# Display compact results: metric_range, trace_range, metric_summary, trace_summary
metric_range, trace_range, metric_summary, trace_summary

```
Out[1]:
```
```python
# Summarize the previous execution results in plain English (compact).
# This cell assumes metric_range, trace_range, metric_summary, trace_summary exist in the kernel.

# Build summary string
mr_min_epoch = int(metric_range['min_timestamp_epoch'])
mr_max_epoch = int(metric_range['max_timestamp_epoch'])
mr_min_local = str(metric_range['min_timestamp_local'])
mr_max_local = str(metric_range['max_timestamp_local'])

tr_min_epoch = int(trace_range['min_timestamp_epoch'])
tr_max_epoch = int(trace_range['max_timestamp_epoch'])
tr_min_local = str(trace_range['min_timestamp_local'])
tr_max_local = str(trace_range['max_timestamp_local'])

# Top 3 metric groups (compact)
top_metrics = []
for _, r in metric_summary.head(3).iterrows():
    top_metrics.append(
        f"{r['cmdb_id']} | {r['kpi_name']} | count={int(r['count'])} | p95={r['p95']} | p50={r['p50']} | p5={r['p5']}"
    )

# Top 3 trace groups (compact)
top_traces = []
for _, r in trace_summary.head(3).iterrows():
    top_traces.append(
        f"{r['cmdb_id']} | {r['trace_name']} | count={int(r['count'])} | p95={r['p95']} | p50={r['p50']} | p5={r['p5']}"
    )

summary = (
    f"metric.csv timestamp range: epoch [{mr_min_epoch}, {mr_max_epoch}], local [{mr_min_local} to {mr_max_local}].\n"
    f"trace.csv timestamp range: epoch [{tr_min_epoch}, {tr_max_epoch}], local [{tr_min_local} to {tr_max_local}].\n\n"
    "Top metric groups (3 shown of top 50 by count):\n  - " + "\n  - ".join(top_metrics) + "\n\n"
    "Top trace groups (3 shown of top 50 by count):\n  - " + "\n  - ".join(top_traces) + "\n\n"
    "Note: Full grouped tables were produced (top 50 rows each). The above is a concise summary."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(min_timestamp_epoch                   1590429600
max_timestamp_epoch                   1590433200
min_timestamp_local    2020-05-26 02:00:00+08:00
max_timestamp_local    2020-05-26 03:00:00+08:00
dtype: object, min_timestamp_epoch                   1590431400
max_timestamp_epoch                   1590433140
min_timestamp_local    2020-05-26 02:30:00+08:00
max_timestamp_local    2020-05-26 02:59:00+08:00
dtype: object,      cmdb_id              kpi_name  count        p95        p90        p50         p5
1256  os_012  Processor_load_5_min     61   0.021667   0.020000   0.008333   0.001667
5     db_001          CPU_free_pct     61  99.268128  99.215812  98.974613  98.070569
976   os_007           Disk_rd_ios     61   0.000000   0.000000   0.000000   0.000000
1638  os_020          Disk_io_util     61   0.005500   0.000000   0.000000   0.000000
95    db_003          CPU_free_pct     61  99.053367  98.962696  97.629333  96.878581
...      ...                   ...    ...        ...        ...        ...        ...
1547  os_018             ICMP_ping     60   1.000000   1.000000   1.000000   1.000000
1529  os_018       CPU_iowait_time     60   0.031307   0.019831   0.003480   0.001356
816   os_004       CPU_system_time     60   0.622909   0.612008   0.567137   0.513898
1485  os_017          Disk_io_util     60   0.000000   0.000000   0.000000   0.000000
784   os_003      Memory_available     60   8.640558   8.640286   8.639065   8.636134

[50 rows x 7 columns],        cmdb_id                                       trace_name  count          p95          p90          p50          p5
0   docker_001    trace.from_os_022.remoteprocess.duration_mean     30     5.960065     3.732472     0.436027    0.353365
1   docker_001     trace.from_os_022.remoteprocess.duration_p95     30    13.714127     7.366600     1.355850    0.802315
2   docker_001       trace.from_os_022.remoteprocess.error_rate     30     0.000000     0.000000     0.000000    0.000000
3   docker_001        trace.from_os_022.remoteprocess.row_count     30   134.100000   132.100000   107.500000   54.550000
4   docker_001                     trace.self.csf.duration_mean     30     1.326004     0.857070     0.049334    0.041312
..         ...                                              ...    ...          ...          ...          ...         ...
45  docker_002                    trace.self.local.duration_p95     30     0.096000     0.096000     0.092000    0.008000
46  docker_002                      trace.self.local.error_rate     30     0.000000     0.000000     0.000000    0.000000
47  docker_002                       trace.self.local.row_count     30  1807.400000  1741.600000  1603.000000  734.300000
48  docker_002  trace.to_docker_007.remoteprocess.duration_mean     30     0.030991     0.028984     0.027399    0.025535
49  docker_002   trace.to_docker_007.remoteprocess.duration_p95     30     0.053490     0.045740     0.038750    0.036135

[50 rows x 7 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables if present, otherwise load metric.csv
try:
    df_metric  # noqa: F821
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Use metric_summary from kernel if exists; otherwise compute global thresholds from entire metric series
if 'metric_summary' in globals():
    thresholds = metric_summary.copy()
else:
    # compute global p95/p5/p50 for each cmdb_id+kpi_name as fallback (must compute BEFORE filtering)
    thresholds = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(p95=lambda s: s.quantile(0.95),
             p90=lambda s: s.quantile(0.90),
             p50='median',
             p5=lambda s: s.quantile(0.05))
        .reset_index()
    )
    # keep only needed p95/p5
    thresholds = thresholds[['cmdb_id', 'kpi_name', 'p95', 'p5']]

# Ensure thresholds has p95 and p5 columns
if 'p95' not in thresholds.columns or 'p5' not in thresholds.columns:
    # try to locate columns with those names if they were named differently
    raise RuntimeError("metric_summary must contain 'p95' and 'p5' columns as global thresholds.")

# Incident window in UTC+8
tz = pytz.timezone('Asia/Shanghai')
start_local = pd.Timestamp("2020-05-26 02:30:00", tz=tz)
end_local   = pd.Timestamp("2020-05-26 03:00:00", tz=tz)
start_ts = int(start_local.timestamp())
end_ts   = int(end_local.timestamp())

# Filter metric rows within the incident window (timestamps are in seconds)
filtered_metric_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# Merge global thresholds into the filtered window (join on cmdb_id & kpi_name)
df_with_thresh = filtered_metric_window.merge(
    thresholds[['cmdb_id', 'kpi_name', 'p95', 'p5']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Keep full merged table in kernel variable for later use
df_metric_with_thresh = df_with_thresh.copy()

# Detect 'high' anomalies (value >= p95) and aggregate
high_mask = df_metric_with_thresh['p95'].notna() & (df_metric_with_thresh['value'] >= df_metric_with_thresh['p95'])
high_df = df_metric_with_thresh[high_mask].copy()
if not high_df.empty:
    high_agg = (
        high_df
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(
            anomaly_type=('value', lambda s: 'high'),
            earliest_anomaly_timestamp=('timestamp', 'min'),
            anomaly_count=('value', 'count'),
            max_value=('value', 'max'),
            min_value=('value', 'min'),
            threshold_used=('p95', 'first')
        )
    )
    # pct_breach uses max_value vs threshold_used
    high_agg['pct_breach'] = np.where(
        high_agg['threshold_used'] == 0,
        np.nan,
        (high_agg['max_value'] - high_agg['threshold_used']).abs() / high_agg['threshold_used']
    )
else:
    high_agg = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_type','earliest_anomaly_timestamp',
        'anomaly_count','max_value','min_value','threshold_used','pct_breach'
    ])

# Detect 'low' anomalies (value <= p5) and aggregate
low_mask = df_metric_with_thresh['p5'].notna() & (df_metric_with_thresh['value'] <= df_metric_with_thresh['p5'])
low_df = df_metric_with_thresh[low_mask].copy()
if not low_df.empty:
    low_agg = (
        low_df
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(
            anomaly_type=('value', lambda s: 'low'),
            earliest_anomaly_timestamp=('timestamp', 'min'),
            anomaly_count=('value', 'count'),
            max_value=('value', 'max'),
            min_value=('value', 'min'),
            threshold_used=('p5', 'first')
        )
    )
    # pct_breach uses min_value vs threshold_used
    low_agg['pct_breach'] = np.where(
        low_agg['threshold_used'] == 0,
        np.nan,
        (low_agg['min_value'] - low_agg['threshold_used']).abs() / low_agg['threshold_used']
    )
else:
    low_agg = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_type','earliest_anomaly_timestamp',
        'anomaly_count','max_value','min_value','threshold_used','pct_breach'
    ])

# Combine high and low aggregates
anomalies_agg = pd.concat([high_agg, low_agg], ignore_index=True, sort=False)

# Convert earliest_anomaly_timestamp to localized human-readable string in UTC+8 for compact display
if not anomalies_agg.empty:
    anomalies_agg['earliest_anomaly_local'] = anomalies_agg['earliest_anomaly_timestamp'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    )
    # Round numeric columns for compactness
    anomalies_agg['pct_breach'] = anomalies_agg['pct_breach'].round(6)
    anomalies_agg['max_value'] = anomalies_agg['max_value'].round(6)
    anomalies_agg['min_value'] = anomalies_agg['min_value'].round(6)
    anomalies_agg['threshold_used'] = anomalies_agg['threshold_used'].round(6)

# Sort by anomaly_count desc then pct_breach desc, limit top 20
anomalies_top20 = anomalies_agg.sort_values(['anomaly_count', 'pct_breach'], ascending=[False, False]).head(20)

# Display compact columns only
result = anomalies_top20[[
    'cmdb_id', 'kpi_name', 'anomaly_type', 'earliest_anomaly_local',
    'anomaly_count', 'max_value', 'min_value', 'threshold_used', 'pct_breach'
]]

result

```
Out[2]:
```
```python
# Build a concise plain-English summary of the anomaly aggregation results already in the kernel.
# Reuse the 'result' DataFrame produced earlier (top aggregated anomaly groups).
try:
    df_res = result  # compact aggregated anomaly summary (top 20)
except NameError:
    import pandas as pd
    df_res = pd.DataFrame()  # fallback empty

# Prepare summary details
total_groups = int(df_res.shape[0])
window_info = "2020-05-26 02:30:00+08:00 to 2020-05-26 03:00:00+08:00"

top_lines = []
for _, r in df_res.head(3).iterrows():
    top_lines.append(
        f"{r['cmdb_id']} | {r['kpi_name']} | {r['anomaly_type']} | earliest={r['earliest_anomaly_local']} | "
        f"count={int(r['anomaly_count'])} | max={r['max_value']} | min={r['min_value']} | "
        f"threshold={r['threshold_used']} | pct_breach={r['pct_breach']}"
    )

note_lines = []
# detect any rows with NaN pct_breach indicating zero threshold
if not df_res.empty and df_res['pct_breach'].isna().any():
    note_lines.append("Some metrics have threshold_used == 0, so pct_breach is NaN for those (e.g., disk I/O metrics).")
# common observation about Agent_ping/ICMP_ping
if not df_res.empty and ((df_res['kpi_name'].str.contains('Agent_ping|ICMP_ping')).any()):
    note_lines.append("Several Agent_ping/ICMP_ping groups show full-window anomalies (count ~30) but zero breach magnitude.")

summary = (
    f"Filtered metric.csv to incident window {window_info}.\n"
    f"Aggregated anomaly groups returned: {total_groups} (top 20 shown).\n\n"
    "Top 3 aggregated anomaly groups:\n  - " + "\n  - ".join(top_lines) + "\n\n"
    + ("Notes: " + " ".join(note_lines) if note_lines else "")
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         kpi_name anomaly_type    earliest_anomaly_local  anomaly_count  max_value  min_value  threshold_used  pct_breach
84  osb_001  app.succee_rate          low  2020-05-26 02:30:00+0800             31        1.0     0.9947             1.0      0.0053
14   os_007      Disk_rd_ios         high  2020-05-26 02:30:00+0800             31        0.0     0.0000             0.0         NaN
50   os_007      Disk_rd_ios          low  2020-05-26 02:30:00+0800             31        0.0     0.0000             0.0         NaN
79   os_020     Disk_io_util          low  2020-05-26 02:30:00+0800             31        0.0     0.0000             0.0         NaN
6    os_004       Agent_ping         high  2020-05-26 02:30:00+0800             30        1.0     1.0000             1.0      0.0000
19   os_017       Agent_ping         high  2020-05-26 02:30:00+0800             30        1.0     1.0000             1.0      0.0000
25   os_018       Agent_ping         high  2020-05-26 02:30:00+0800             30        1.0     1.0000             1.0      0.0000
32   os_018        ICMP_ping         high  2020-05-26 02:30:00+0800             30        1.0     1.0000             1.0      0.0000
35  osb_001  app.succee_rate         high  2020-05-26 02:30:00+0800             30        1.0     1.0000             1.0      0.0000
42   os_004       Agent_ping          low  2020-05-26 02:30:00+0800             30        1.0     1.0000             1.0      0.0000
56   os_017       Agent_ping          low  2020-05-26 02:30:00+0800             30        1.0     1.0000             1.0      0.0000
65   os_018       Agent_ping          low  2020-05-26 02:30:00+0800             30        1.0     1.0000             1.0      0.0000
76   os_018        ICMP_ping          low  2020-05-26 02:30:00+0800             30        1.0     1.0000             1.0      0.0000
22   os_017     Disk_io_util         high  2020-05-26 02:30:00+0800             30        0.0     0.0000             0.0         NaN
23   os_017      Disk_rd_ios         high  2020-05-26 02:30:00+0800             30        0.0     0.0000             0.0         NaN
27   os_018     Disk_io_util         high  2020-05-26 02:30:00+0800             30        0.0     0.0000             0.0         NaN
28   os_018      Disk_rd_ios         high  2020-05-26 02:30:00+0800             30        0.0     0.0000             0.0         NaN
29   os_018      Disk_rd_kbs         high  2020-05-26 02:30:00+0800             30        0.0     0.0000             0.0         NaN
59   os_017     Disk_io_util          low  2020-05-26 02:30:00+0800             30        0.0     0.0000             0.0         NaN
60   os_017      Disk_rd_ios          low  2020-05-26 02:30:00+0800             30        0.0     0.0000             0.0         NaN```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# reuse df_trace if present, otherwise load
try:
    df_trace  # noqa: F821
except NameError:
    df_trace = pd.read_csv("trace.csv")

# reuse trace_summary (global thresholds) if present; otherwise compute BEFORE filtering
if 'trace_summary' in globals():
    trace_thresholds = trace_summary.copy()
else:
    trace_thresholds = (
        df_trace
        .groupby(['cmdb_id', 'trace_name'])['value']
        .agg(p95=lambda s: s.quantile(0.95),
             p90=lambda s: s.quantile(0.90),
             p50='median',
             p5=lambda s: s.quantile(0.05))
        .reset_index()
    )

# Ensure required threshold columns exist
if 'p95' not in trace_thresholds.columns or 'p5' not in trace_thresholds.columns:
    raise RuntimeError("trace_summary must contain 'p95' and 'p5' columns as global thresholds.")

# Incident window (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_local = pd.Timestamp("2020-05-26 02:30:00", tz=tz)
end_local   = pd.Timestamp("2020-05-26 03:00:00", tz=tz)
start_ts = int(start_local.timestamp())
end_ts = int(end_local.timestamp())

# Filter trace rows within the incident window
filtered_trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# Merge global thresholds into the filtered window (join on cmdb_id & trace_name)
df_trace_with_thresh = filtered_trace_window.merge(
    trace_thresholds[['cmdb_id', 'trace_name', 'p95', 'p5']],
    on=['cmdb_id', 'trace_name'],
    how='left'
)

# Keep full merged table in kernel variable for later inspection
trace_window_with_thresholds = df_trace_with_thresh.copy()

# Detect high anomalies (value >= p95)
high_mask = df_trace_with_thresh['p95'].notna() & (df_trace_with_thresh['value'] >= df_trace_with_thresh['p95'])
high_df = df_trace_with_thresh[high_mask].copy()
if not high_df.empty:
    high_agg = (
        high_df
        .groupby(['cmdb_id', 'trace_name'], as_index=False)
        .agg(
            anomaly_type=('value', lambda s: 'high'),
            earliest_anomaly_timestamp=('timestamp', 'min'),
            anomaly_count=('value', 'count'),
            max_value=('value', 'max'),
            min_value=('value', 'min'),
            threshold_used=('p95', 'first')
        )
    )
    high_agg['pct_breach'] = np.where(
        high_agg['threshold_used'] == 0,
        np.nan,
        (high_agg['max_value'] - high_agg['threshold_used']).abs() / high_agg['threshold_used']
    )
else:
    high_agg = pd.DataFrame(columns=[
        'cmdb_id','trace_name','anomaly_type','earliest_anomaly_timestamp',
        'anomaly_count','max_value','min_value','threshold_used','pct_breach'
    ])

# Detect low anomalies (value <= p5)
low_mask = df_trace_with_thresh['p5'].notna() & (df_trace_with_thresh['value'] <= df_trace_with_thresh['p5'])
low_df = df_trace_with_thresh[low_mask].copy()
if not low_df.empty:
    low_agg = (
        low_df
        .groupby(['cmdb_id', 'trace_name'], as_index=False)
        .agg(
            anomaly_type=('value', lambda s: 'low'),
            earliest_anomaly_timestamp=('timestamp', 'min'),
            anomaly_count=('value', 'count'),
            max_value=('value', 'max'),
            min_value=('value', 'min'),
            threshold_used=('p5', 'first')
        )
    )
    low_agg['pct_breach'] = np.where(
        low_agg['threshold_used'] == 0,
        np.nan,
        (low_agg['min_value'] - low_agg['threshold_used']).abs() / low_agg['threshold_used']
    )
else:
    low_agg = pd.DataFrame(columns=[
        'cmdb_id','trace_name','anomaly_type','earliest_anomaly_timestamp',
        'anomaly_count','max_value','min_value','threshold_used','pct_breach'
    ])

# Combine high and low aggregates
trace_anomalies_agg = pd.concat([high_agg, low_agg], ignore_index=True, sort=False)

# Convert earliest_anomaly_timestamp to localized human-readable string in UTC+8 and round numeric columns
if not trace_anomalies_agg.empty:
    trace_anomalies_agg['earliest_anomaly_local'] = trace_anomalies_agg['earliest_anomaly_timestamp'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    )
    trace_anomalies_agg['pct_breach'] = trace_anomalies_agg['pct_breach'].round(6)
    trace_anomalies_agg['max_value'] = trace_anomalies_agg['max_value'].round(6)
    trace_anomalies_agg['min_value'] = trace_anomalies_agg['min_value'].round(6)
    trace_anomalies_agg['threshold_used'] = trace_anomalies_agg['threshold_used'].round(6)

# Prepare final compact result sorted by anomaly_count desc, pct_breach desc, top 20
trace_anomalies_top20 = trace_anomalies_agg.sort_values(['anomaly_count', 'pct_breach'], ascending=[False, False]).head(20)

# Select requested columns for compact display
trace_anomalies_result = trace_anomalies_top20[[
    'cmdb_id', 'trace_name', 'anomaly_type', 'earliest_anomaly_local',
    'anomaly_count', 'max_value', 'min_value', 'threshold_used', 'pct_breach'
]]

# Display compact aggregated result (and keep intermediate full merged table as trace_window_with_thresholds)
trace_anomalies_result, trace_window_with_thresholds.shape

```
Out[3]:
```
```python
# Summarize trace anomaly aggregation results in plain English (compact).
try:
    tbl = trace_anomalies_result  # top aggregated anomalies (compact)
    full_shape = trace_window_with_thresholds.shape  # filtered window full table shape
except NameError:
    tbl = None
    full_shape = (0, 0)

tz_note = "all times in UTC+8"
window = "2020-05-26 02:30:00+08:00 through 2020-05-26 03:00:00+08:00"

if tbl is None or tbl.empty:
    summary = "No trace anomaly aggregates available for the incident window."
else:
    total_filtered_rows = int(full_shape[0])
    total_reported_groups = int(tbl.shape[0])
    # build top lines (up to 6) for illustration
    top_lines = []
    for _, r in tbl.head(6).iterrows():
        top_lines.append(
            f"{r['cmdb_id']} | {r['trace_name']} | {r['anomaly_type']} | count={int(r['anomaly_count'])} | "
            f"earliest={r['earliest_anomaly_local']} | max={r['max_value']} | min={r['min_value']} | "
            f"threshold={r['threshold_used']} | pct_breach={r['pct_breach']}"
        )
    nan_pct = tbl['pct_breach'].isna().any()
    note_parts = []
    if nan_pct:
        note_parts.append(
            "Many groups have threshold_used == 0 and metric values == 0, so pct_breach is NaN (division by zero)."
        )
    note_parts.append(
        "Several 'error_rate' and other trace metrics for docker_001/docker_002 are flagged across the entire window "
        "(anomaly_count=30, earliest=2020-05-26 02:30:00+0800), but values are zero and equal to thresholds."
    )
    summary = (
        f"Trace anomaly summary for window {window} ({tz_note}):\n"
        f"- Filtered trace rows in window: {total_filtered_rows}\n"
        f"- Aggregated anomaly groups returned (top shown, total={total_reported_groups}):\n  - " + "\n  - ".join(top_lines) + "\n"
        f"- Notes: " + " ".join(note_parts)
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                    trace_name anomaly_type    earliest_anomaly_local  anomaly_count  max_value  min_value  threshold_used  pct_breach
2   docker_001    trace.from_os_022.remoteprocess.error_rate         high  2020-05-26 02:30:00+0800             30   0.000000        0.0             0.0         NaN
6   docker_001                     trace.self.csf.error_rate         high  2020-05-26 02:30:00+0800             30   0.000000        0.0             0.0         NaN
10  docker_001               trace.self.flyremote.error_rate         high  2020-05-26 02:30:00+0800             30   0.000000        0.0             0.0         NaN
14  docker_001                    trace.self.jdbc.error_rate         high  2020-05-26 02:30:00+0800             30   0.000000        0.0             0.0         NaN
18  docker_001                   trace.self.local.error_rate         high  2020-05-26 02:30:00+0800             30   0.000000        0.0             0.0         NaN
22  docker_001  trace.to_docker_007.remoteprocess.error_rate         high  2020-05-26 02:30:00+0800             30   0.000000        0.0             0.0         NaN
26  docker_001  trace.to_docker_008.remoteprocess.error_rate         high  2020-05-26 02:30:00+0800             30   0.000000        0.0             0.0         NaN
30  docker_002    trace.from_os_022.remoteprocess.error_rate         high  2020-05-26 02:30:00+0800             30   0.000000        0.0             0.0         NaN
34  docker_002                     trace.self.csf.error_rate         high  2020-05-26 02:30:00+0800             30   0.004878        0.0             0.0         NaN
38  docker_002               trace.self.flyremote.error_rate         high  2020-05-26 02:30:00+0800             30   0.000000        0.0             0.0         NaN
42  docker_002                    trace.self.jdbc.error_rate         high  2020-05-26 02:30:00+0800             30   0.000000        0.0             0.0         NaN
46  docker_002                   trace.self.local.error_rate         high  2020-05-26 02:30:00+0800             30   0.000000        0.0             0.0         NaN
52  docker_001    trace.from_os_022.remoteprocess.error_rate          low  2020-05-26 02:30:00+0800             30   0.000000        0.0             0.0         NaN
56  docker_001                     trace.self.csf.error_rate          low  2020-05-26 02:30:00+0800             30   0.000000        0.0             0.0         NaN
60  docker_001               trace.self.flyremote.error_rate          low  2020-05-26 02:30:00+0800             30   0.000000        0.0             0.0         NaN
64  docker_001                    trace.self.jdbc.error_rate          low  2020-05-26 02:30:00+0800             30   0.000000        0.0             0.0         NaN
68  docker_001                   trace.self.local.error_rate          low  2020-05-26 02:30:00+0800             30   0.000000        0.0             0.0         NaN
72  docker_001  trace.to_docker_007.remoteprocess.error_rate          low  2020-05-26 02:30:00+0800             30   0.000000        0.0             0.0         NaN
76  docker_001  trace.to_docker_008.remoteprocess.error_rate          low  2020-05-26 02:30:00+0800             30   0.000000        0.0             0.0         NaN
80  docker_002    trace.from_os_022.remoteprocess.error_rate          low  2020-05-26 02:30:00+0800             30   0.000000        0.0             0.0         NaN, (5520, 6))```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing df_trace and trace_summary if available, otherwise load/compute
try:
    df_trace  # noqa: F821
except NameError:
    df_trace = pd.read_csv("trace.csv")

if 'trace_summary' in globals():
    ts_thresh = trace_summary.copy()
else:
    ts_thresh = (
        df_trace
        .groupby(['cmdb_id', 'trace_name'])['value']
        .agg(p95=lambda s: s.quantile(0.95),
             p5=lambda s: s.quantile(0.05))
        .reset_index()
    )

# Select trace groups with global p95 > 0 and trace_name contains 'duration_mean' or 'duration_p95'
mask_duration = ts_thresh['trace_name'].str.contains('duration_mean|duration_p95', regex=True, na=False)
selected_thresholds = ts_thresh[mask_duration & (ts_thresh['p95'] > 0)][['cmdb_id', 'trace_name', 'p95']].copy()

# Incident window in UTC+8
tz = pytz.timezone('Asia/Shanghai')
start_local = pd.Timestamp("2020-05-26 02:30:00", tz=tz)
end_local   = pd.Timestamp("2020-05-26 03:00:00", tz=tz)
start_ts = int(start_local.timestamp())
end_ts = int(end_local.timestamp())

# Filter trace rows to the incident window
trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# Join per-row with the group's global p95 (only for selected duration groups)
trace_joined = trace_window.merge(
    selected_thresholds,
    on=['cmdb_id', 'trace_name'],
    how='inner'  # inner: keep only rows for the selected duration groups
)

# Keep full filtered-joined table for follow-up inspection
trace_duration_joined = trace_joined.copy()  # kernel variable stored

# Identify rows with value >= global_p95 (global p95 > 0 ensured above)
trace_duration_anomalies = trace_duration_joined[trace_duration_joined['value'] >= trace_duration_joined['p95']].copy()
# compute pct_breach safely (p95 > 0 by selection)
trace_duration_anomalies['pct_breach'] = (trace_duration_anomalies['value'] - trace_duration_anomalies['p95']) / trace_duration_anomalies['p95']

# Store filtered anomaly rows for later inspection
trace_duration_anomaly_rows = trace_duration_anomalies.copy()  # kernel variable stored

# Aggregate by cmdb_id and trace_name
if not trace_duration_anomalies.empty:
    agg = (
        trace_duration_anomalies
        .groupby(['cmdb_id', 'trace_name'], as_index=False)
        .agg(
            earliest_anomaly_timestamp=('timestamp', 'min'),
            anomaly_count=('value', 'count'),
            max_value=('value', 'max'),
            threshold_p95=('p95', 'first'),
            pct_breach_max=('pct_breach', 'max')
        )
    )
    # Convert earliest_anomaly_timestamp to localized human-readable string in UTC+8
    agg['earliest_anomaly_local'] = agg['earliest_anomaly_timestamp'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    )
    # Round numeric columns for compactness
    agg['pct_breach_max'] = agg['pct_breach_max'].round(6)
    agg['max_value'] = agg['max_value'].round(6)
    agg['threshold_p95'] = agg['threshold_p95'].round(6)
else:
    agg = pd.DataFrame(columns=[
        'cmdb_id','trace_name','earliest_anomaly_timestamp','anomaly_count',
        'max_value','threshold_p95','pct_breach_max','earliest_anomaly_local'
    ])

# Sort by pct_breach_max desc then anomaly_count desc and limit to top 20
trace_duration_anomaly_agg = agg.sort_values(['pct_breach_max', 'anomaly_count'], ascending=[False, False]).head(20)

# Display compact aggregated result and count of anomaly rows
trace_duration_anomaly_agg[['cmdb_id','trace_name','earliest_anomaly_local','anomaly_count','max_value','threshold_p95','pct_breach_max']], trace_duration_anomaly_rows.shape

```
Out[4]:
```
```python
# Summarize the duration-based trace anomalies found (uses kernel variables trace_duration_anomaly_agg and trace_duration_anomaly_rows).
try:
    agg = trace_duration_anomaly_agg  # aggregated results (top groups)
    rows = trace_duration_anomaly_rows  # filtered anomaly rows
except NameError:
    agg = None
    rows = None

if agg is None or agg.empty:
    summary = "No duration-based trace anomalies found in the incident window."
else:
    total_groups = int(agg.shape[0])
    total_rows = int(rows.shape[0]) if rows is not None else 0
    window = "2020-05-26 02:30:00+08:00 through 2020-05-26 03:00:00+08:00 (UTC+8)"
    top_lines = []
    for _, r in agg.head(5).iterrows():
        top_lines.append(
            f"{r['cmdb_id']} | {r['trace_name']} | pct_breach_max={r['pct_breach_max']} | "
            f"count={int(r['anomaly_count'])} | threshold_p95={r['threshold_p95']} | max_value={r['max_value']} | earliest={r['earliest_anomaly_local']}"
        )
    summary = (
        f"Selected trace groups: those with global p95>0 and name containing 'duration_mean' or 'duration_p95'.\n"
        f"Incident window: {window}.\n"
        f"Anomalous rows matching criteria: {total_rows} rows.\n"
        f"Aggregated anomalous groups: {total_groups} groups (top 20 computed).\n"
        f"Top examples (up to 5):\n  - " + "\n  - ".join(top_lines) + "\n"
        f"Interpretation: Multiple docker_* duration metrics (notably from_os_022 and remoteprocess traces) exceeded their global p95 by large margins (pct_breach_max ~0.7), "
        f"each observed in a small number of points (anomaly_count=2), indicating short-lived but substantial latency spikes during the window."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                       trace_name    earliest_anomaly_local  anomaly_count  max_value  threshold_p95  pct_breach_max
0   docker_001    trace.from_os_022.remoteprocess.duration_mean  2020-05-26 02:43:00+0800              2  10.165528       5.960065        0.705607
13  docker_001   trace.to_docker_008.remoteprocess.duration_p95  2020-05-26 02:43:00+0800              2   6.091800       3.574007        0.704473
15  docker_002     trace.from_os_022.remoteprocess.duration_p95  2020-05-26 02:43:00+0800              2  17.442800      10.376052        0.681063
2   docker_001                     trace.self.csf.duration_mean  2020-05-26 02:43:00+0800              2   2.208893       1.326004        0.665827
14  docker_002    trace.from_os_022.remoteprocess.duration_mean  2020-05-26 02:43:00+0800              2   7.375533       4.497155        0.640044
16  docker_002                     trace.self.csf.duration_mean  2020-05-26 02:43:00+0800              2   1.782382       1.089410        0.636098
7   docker_001                     trace.self.jdbc.duration_p95  2020-05-26 02:43:00+0800              2   0.008000       0.005247        0.524681
17  docker_002                      trace.self.csf.duration_p95  2020-05-26 02:43:00+0800              2   5.858400       4.002280        0.463766
12  docker_001  trace.to_docker_008.remoteprocess.duration_mean  2020-05-26 02:43:00+0800              2   2.437239       1.810689        0.346029
3   docker_001                      trace.self.csf.duration_p95  2020-05-26 02:43:00+0800              2   6.063800       4.709650        0.287527
11  docker_001   trace.to_docker_007.remoteprocess.duration_p95  2020-05-26 02:43:00+0800              2   0.061000       0.047495        0.284346
1   docker_001     trace.from_os_022.remoteprocess.duration_p95  2020-05-26 02:43:00+0800              2  16.797750      13.714127        0.224850
25  docker_002   trace.to_docker_007.remoteprocess.duration_p95  2020-05-26 02:43:00+0800              2   0.065000       0.053490        0.215180
5   docker_001                trace.self.flyremote.duration_p95  2020-05-26 02:40:00+0800              2   0.090100       0.078290        0.150849
10  docker_001  trace.to_docker_007.remoteprocess.duration_mean  2020-05-26 02:43:00+0800              2   0.032984       0.028832        0.144007
4   docker_001               trace.self.flyremote.duration_mean  2020-05-26 02:40:00+0800              2   0.012109       0.010642        0.137850
18  docker_002               trace.self.flyremote.duration_mean  2020-05-26 02:46:00+0800              2   0.015505       0.013836        0.120627
20  docker_002                    trace.self.jdbc.duration_mean  2020-05-26 02:48:00+0800              2   0.004380       0.004142        0.057460
8   docker_001                   trace.self.local.duration_mean  2020-05-26 02:54:00+0800              2   0.015678       0.014865        0.054692
24  docker_002  trace.to_docker_007.remoteprocess.duration_mean  2020-05-26 02:43:00+0800              2   0.032392       0.030991        0.045207, (55, 6))```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing anomaly rows if present
try:
    trace_duration_anomaly_rows  # produced earlier
except NameError:
    # fallback: empty DataFrame with expected columns
    trace_duration_anomaly_rows = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','p95','pct_breach'])

# Ensure numeric columns exist
for col in ['value','p95','pct_breach']:
    if col not in trace_duration_anomaly_rows.columns:
        trace_duration_anomaly_rows[col] = np.nan

# Ensure timestamp is integer type for sorting
if not trace_duration_anomaly_rows.empty:
    trace_duration_anomaly_rows['timestamp'] = trace_duration_anomaly_rows['timestamp'].astype(int)

# Create localized timestamp in UTC+8
tz = pytz.timezone('Asia/Shanghai')
if not trace_duration_anomaly_rows.empty:
    trace_duration_anomaly_rows['timestamp_local'] = pd.to_datetime(
        trace_duration_anomaly_rows['timestamp'], unit='s', utc=True
    ).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')
else:
    trace_duration_anomaly_rows['timestamp_local'] = []

# Compute pct_breach if missing and p95>0
mask_compute = trace_duration_anomaly_rows['pct_breach'].isna() & trace_duration_anomaly_rows['p95'].notna() & (trace_duration_anomaly_rows['p95'] != 0)
if mask_compute.any():
    trace_duration_anomaly_rows.loc[mask_compute, 'pct_breach'] = (
        (trace_duration_anomaly_rows.loc[mask_compute, 'value'] - trace_duration_anomaly_rows.loc[mask_compute, 'p95'])
        / trace_duration_anomaly_rows.loc[mask_compute, 'p95']
    )

# Prepare view with required columns and formatting, sorted by timestamp ascending, up to 50 rows
cols = ['timestamp', 'timestamp_local', 'cmdb_id', 'trace_name', 'value', 'p95', 'pct_breach']
trace_duration_anomaly_rows_view = (
    trace_duration_anomaly_rows.loc[:, cols]
    .rename(columns={'p95': 'threshold_p95'})
    .assign(
        value=lambda df: df['value'].round(6),
        threshold_p95=lambda df: df['threshold_p95'].round(6),
        pct_breach=lambda df: df['pct_breach'].round(6)
    )
    .sort_values('timestamp', ascending=True)
    .head(50)
    .reset_index(drop=True)
)

# Display the result (compact)
trace_duration_anomaly_rows_view

```
Out[5]:
```
import pandas as pd

# Summarize trace_duration_anomaly_rows_view (assumed stored in kernel)
try:
    df_view = trace_duration_anomaly_rows_view  # noqa: F821
except NameError:
    df_view = pd.DataFrame()

if df_view.empty:
    summary = "No duration-anomalous trace rows are available in the kernel."
else:
    n_rows = int(df_view.shape[0])
    # earliest and latest local timestamps (strings already in timestamp_local)
    earliest_local = df_view['timestamp_local'].min()
    latest_local = df_view['timestamp_local'].max()
    # number of distinct trace groups
    n_groups = int(df_view[['cmdb_id', 'trace_name']].drop_duplicates().shape[0])
    # top 5 rows by pct_breach
    top5 = df_view.sort_values('pct_breach', ascending=False).head(5)
    top_lines = []
    for _, r in top5.iterrows():
        top_lines.append(
            f"{r['cmdb_id']} | {r['trace_name']} | ts={r['timestamp_local']} | value={r['value']:.6f} | "
            f"threshold_p95={r['threshold_p95']:.6f} | pct_breach={r['pct_breach']:.6f}"
        )
    summary = (
        f"Returned up to 50 duration-anomalous trace rows (sorted by timestamp ascending).\n"
        f"Rows returned: {n_rows}. Distinct cmdb_id+trace_name groups: {n_groups}.\n"
        f"Timestamp range (local, UTC+8): {earliest_local} through {latest_local}.\n"
        f"Columns included: timestamp (epoch), timestamp_local (UTC+8), cmdb_id, trace_name, value, threshold_p95, pct_breach.\n\n"
        f"Top 5 breaches by pct_breach:\n  - " + "\n  - ".join(top_lines)
    )

summary

The original code execution output of IPython Kernel is also provided below for reference:

timestamp           timestamp_local     cmdb_id                          trace_name     value  threshold_p95  pct_breach
0   1590431640  2020-05-26 02:34:00+0800  docker_001       trace.self.jdbc.duration_mean  0.003535       0.003452    0.024044
1   1590431760  2020-05-26 02:36:00+0800  docker_002        trace.self.jdbc.duration_p95  0.004000       0.004000    0.000000
2   1590431820  2020-05-26 02:37:00+0800  docker_002       trace.self.local.duration_p95  0.096000       0.096000    0.000000
3   1590431940  2020-05-26 02:39:00+0800  docker_002   trace.self.flyremote.duration_p95  0.095000       0.094530    0.004972
4   1590432000  2020-05-26 02:40:00+0800  docker_001  trace.self.flyremote.duration_mean  0.012109       0.010642    0.137850
..         ...                       ...         ...                                 ...       ...            ...         ...
45  1590432900  2020-05-26 02:55:00+0800  docker_001       trace.self.local.duration_p95  0.096000       0.095100    0.009464
46  1590432960  2020-05-26 02:56:00+0800  docker_002  trace.self.flyremote.duration_mean  0.015505       0.013836    0.120627
47  1590432960  2020-05-26 02:56:00+0800  docker_002   trace.self.flyremote.duration_p95  0.094800       0.094530    0.002856
48  1590432960  2020-05-26 02:56:00+0800  docker_002       trace.self.local.duration_p95  0.097000       0.096000    0.010417
49  1590433020  2020-05-26 02:57:00+0800  docker_001  trace.self.flyremote.duration_mean  0.010722       0.010642    0.007517

[50 rows x 7 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# reuse df_trace if present, otherwise load
try:
    df_trace  # noqa: F821
except NameError:
    df_trace = pd.read_csv("trace.csv")

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Define narrow window in local tz and convert to epoch seconds
start_local = pd.Timestamp("2020-05-26 02:40:00", tz=tz)
end_local   = pd.Timestamp("2020-05-26 02:46:00", tz=tz)
start_ts = int(start_local.timestamp())
end_ts = int(end_local.timestamp())

# Filter to window
trace_window_narrow = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# Filter trace_name containing any of the required substrings
pattern = r'(from_os_022|remoteprocess|to_docker)'
mask = trace_window_narrow['trace_name'].str.contains(pattern, regex=True, na=False)
trace_window_callchain_rows = trace_window_narrow[mask].copy()

# Add localized timestamp string in UTC+8
if not trace_window_callchain_rows.empty:
    trace_window_callchain_rows['timestamp_local'] = pd.to_datetime(
        trace_window_callchain_rows['timestamp'], unit='s', utc=True
    ).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')
else:
    trace_window_callchain_rows['timestamp_local'] = []

# Select requested columns and sort ascending, limit to 200 rows
trace_window_callchain_rows = (
    trace_window_callchain_rows.loc[:, ['timestamp', 'timestamp_local', 'cmdb_id', 'trace_name', 'value']]
    .sort_values('timestamp', ascending=True)
    .head(200)
    .reset_index(drop=True)
)

# Aggregation: count per cmdb_id and earliest timestamp_local per cmdb_id
if not trace_window_callchain_rows.empty:
    agg = (
        trace_window_callchain_rows
        .groupby('cmdb_id', as_index=False)
        .agg(
            row_count=('timestamp', 'count'),
            earliest_timestamp_epoch=('timestamp', 'min')
        )
    )
    agg['earliest_timestamp_local'] = pd.to_datetime(agg['earliest_timestamp_epoch'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')
    trace_callchain_agg = agg[['cmdb_id', 'row_count', 'earliest_timestamp_local']]
    trace_callchain_agg_top20 = trace_callchain_agg.sort_values('row_count', ascending=False).head(20).reset_index(drop=True)
else:
    trace_callchain_agg_top20 = pd.DataFrame(columns=['cmdb_id', 'row_count', 'earliest_timestamp_local'])

# Return the table and the compact aggregate summary
trace_window_callchain_rows, trace_callchain_agg_top20

```
Out[6]:
```
```python
# Build a concise plain-English summary of the filtered call-chain rows and compact aggregation.
try:
    rows = trace_window_callchain_rows  # table of filtered rows (up to 200)
    agg = trace_callchain_agg_top20     # aggregated counts per cmdb_id (top 20)
except NameError:
    rows = None
    agg = None

window = "2020-05-26 02:40:00+08:00 through 2020-05-26 02:46:00+08:00 (UTC+8)"

if rows is None or rows.empty:
    summary = "No matching trace rows found in the specified narrow window."
else:
    total_rows = int(rows.shape[0])
    distinct_groups = int(rows[['cmdb_id', 'trace_name']].drop_duplicates().shape[0])
    # prepare top cmdb_id lines (limit to 8 for compactness)
    top_lines = []
    if agg is not None and not agg.empty:
        for _, r in agg.head(8).iterrows():
            top_lines.append(f"{r['cmdb_id']}: count={int(r['row_count'])}, earliest={r['earliest_timestamp_local']}")
    else:
        top_lines = ["(no aggregate available)"]
    summary = (
        f"Filtered trace rows for window {window}.\n"
        f"- Rows returned: {total_rows} (sorted by timestamp ascending, limited to 200).\n"
        f"- Distinct cmdb_id+trace_name groups in these rows: {distinct_groups}.\n"
        f"- Trace names filtered to contain: 'from_os_022', 'remoteprocess', or 'to_docker'.\n"
        f"- Top cmdb_id by row count (up to 8):\n  - " + "\n  - ".join(top_lines) + "\n"
        f"- Example row columns: timestamp (epoch), timestamp_local (UTC+8), cmdb_id, trace_name, value.\n"
        f"Note: Full filtered table stored in kernel variable 'trace_window_callchain_rows'; compact per-cmdb_id aggregate in 'trace_callchain_agg_top20'."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      timestamp           timestamp_local cmdb_id                                       trace_name       value
0    1590432000  2020-05-26 02:40:00+0800  os_021  trace.to_docker_003.remoteprocess.duration_mean    0.344117
1    1590432000  2020-05-26 02:40:00+0800  os_021   trace.to_docker_003.remoteprocess.duration_p95    0.905500
2    1590432000  2020-05-26 02:40:00+0800  os_021     trace.to_docker_003.remoteprocess.error_rate    0.000000
3    1590432000  2020-05-26 02:40:00+0800  os_021      trace.to_docker_003.remoteprocess.row_count  111.000000
4    1590432000  2020-05-26 02:40:00+0800  os_021  trace.to_docker_004.remoteprocess.duration_mean    0.314690
..          ...                       ...     ...                                              ...         ...
195  1590432120  2020-05-26 02:42:00+0800  os_021  trace.to_docker_004.remoteprocess.duration_mean    0.221980
196  1590432120  2020-05-26 02:42:00+0800  os_021      trace.to_docker_003.remoteprocess.row_count   73.000000
197  1590432120  2020-05-26 02:42:00+0800  os_021     trace.to_docker_003.remoteprocess.error_rate    0.000000
198  1590432120  2020-05-26 02:42:00+0800  os_021   trace.to_docker_003.remoteprocess.duration_p95    0.949400
199  1590432120  2020-05-26 02:42:00+0800  os_021  trace.to_docker_003.remoteprocess.duration_mean    0.314274

[200 rows x 5 columns],       cmdb_id  row_count  earliest_timestamp_local
0  docker_001         24  2020-05-26 02:40:00+0800
1  docker_002         24  2020-05-26 02:40:00+0800
2  docker_004         24  2020-05-26 02:40:00+0800
3  docker_005         24  2020-05-26 02:40:00+0800
4  docker_007         24  2020-05-26 02:40:00+0800
5      os_021         24  2020-05-26 02:40:00+0800
6  docker_008         24  2020-05-26 02:40:00+0800
7  docker_003         16  2020-05-26 02:40:00+0800
8      os_022         16  2020-05-26 02:40:00+0800)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing df_metric if present
try:
    df_metric  # noqa: F821
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Use precomputed global thresholds from metric_summary if available; otherwise compute BEFORE filtering
if 'metric_summary' in globals():
    thresholds = metric_summary.copy()
else:
    # compute global thresholds from entire metric series (must be done before filtering)
    thresholds = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(p95=lambda s: s.quantile(0.95),
             p90=lambda s: s.quantile(0.90),
             p50='median',
             p5=lambda s: s.quantile(0.05))
        .reset_index()
    )

# Ensure p95 and p5 columns exist
for col in ['p95', 'p5']:
    if col not in thresholds.columns:
        thresholds[col] = np.nan

# Incident window in UTC+8
tz = pytz.timezone('Asia/Shanghai')
start_local = pd.Timestamp("2020-05-26 02:30:00", tz=tz)
end_local   = pd.Timestamp("2020-05-26 03:00:00", tz=tz)
start_ts = int(start_local.timestamp())
end_ts = int(end_local.timestamp())

# Filter metric.csv to cmdb_id == 'os_022' and the incident window
os022_window = df_metric[
    (df_metric['cmdb_id'] == 'os_022') &
    (df_metric['timestamp'] >= start_ts) &
    (df_metric['timestamp'] <= end_ts)
].copy()

# Join per-row with global thresholds (p95 and p5) using precomputed thresholds
os022_with_thresh = os022_window.merge(
    thresholds[['cmdb_id', 'kpi_name', 'p95', 'p5']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Store raw rows (up to 50) with requested columns and timestamp_local, threshold_p95, threshold_p5
if not os022_with_thresh.empty:
    os022_with_thresh['timestamp_local'] = pd.to_datetime(
        os022_with_thresh['timestamp'], unit='s', utc=True
    ).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')
else:
    os022_with_thresh['timestamp_local'] = []

os022_metric_window_rows = (
    os022_with_thresh
    .loc[:, ['timestamp', 'timestamp_local', 'kpi_name', 'value', 'p95', 'p5']]
    .rename(columns={'p95': 'threshold_p95', 'p5': 'threshold_p5'})
    .sort_values('timestamp', ascending=True)
    .head(50)
    .reset_index(drop=True)
)

# Detect anomalies per row:
# - high: value >= p95 and p95>0
# - low: value <= p5 and p5>0
high_mask = os022_with_thresh['p95'].notna() & (os022_with_thresh['p95'] > 0) & (os022_with_thresh['value'] >= os022_with_thresh['p95'])
low_mask  = os022_with_thresh['p5'].notna()  & (os022_with_thresh['p5'] > 0)  & (os022_with_thresh['value'] <= os022_with_thresh['p5'])

high_df = os022_with_thresh[high_mask].copy()
low_df  = os022_with_thresh[low_mask].copy()

# Aggregate high anomalies
if not high_df.empty:
    high_agg = (
        high_df
        .groupby(['kpi_name'], as_index=False)
        .agg(
            cmdb_id=('cmdb_id', 'first'),
            anomaly_type=('value', lambda s: 'high'),
            earliest_anomaly_timestamp=('timestamp', 'min'),
            anomaly_count=('value', 'count'),
            max_value=('value', 'max'),
            min_value=('value', 'min'),
            threshold_used=('p95', 'first')
        )
    )
    # pct_breach = (max_value - threshold)/threshold
    high_agg['pct_breach'] = np.where(
        high_agg['threshold_used'] == 0,
        np.nan,
        (high_agg['max_value'] - high_agg['threshold_used']) / high_agg['threshold_used']
    )
else:
    high_agg = pd.DataFrame(columns=['kpi_name','cmdb_id','anomaly_type','earliest_anomaly_timestamp','anomaly_count','max_value','min_value','threshold_used','pct_breach'])

# Aggregate low anomalies
if not low_df.empty:
    low_agg = (
        low_df
        .groupby(['kpi_name'], as_index=False)
        .agg(
            cmdb_id=('cmdb_id', 'first'),
            anomaly_type=('value', lambda s: 'low'),
            earliest_anomaly_timestamp=('timestamp', 'min'),
            anomaly_count=('value', 'count'),
            max_value=('value', 'max'),
            min_value=('value', 'min'),
            threshold_used=('p5', 'first')
        )
    )
    # pct_breach = (threshold - min_value)/threshold
    low_agg['pct_breach'] = np.where(
        low_agg['threshold_used'] == 0,
        np.nan,
        (low_agg['threshold_used'] - low_agg['min_value']) / low_agg['threshold_used']
    )
else:
    low_agg = pd.DataFrame(columns=['kpi_name','cmdb_id','anomaly_type','earliest_anomaly_timestamp','anomaly_count','max_value','min_value','threshold_used','pct_breach'])

# Combine and format aggregated results
os022_metric_anomaly_agg = pd.concat([high_agg, low_agg], ignore_index=True, sort=False)

if not os022_metric_anomaly_agg.empty:
    # convert earliest timestamp to local string and round numbers
    os022_metric_anomaly_agg['earliest_anomaly_local'] = os022_metric_anomaly_agg['earliest_anomaly_timestamp'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    )
    os022_metric_anomaly_agg['pct_breach'] = os022_metric_anomaly_agg['pct_breach'].round(6)
    os022_metric_anomaly_agg['max_value'] = os022_metric_anomaly_agg['max_value'].round(6)
    os022_metric_anomaly_agg['min_value'] = os022_metric_anomaly_agg['min_value'].round(6)
    os022_metric_anomaly_agg['threshold_used'] = os022_metric_anomaly_agg['threshold_used'].round(6)

# Sort aggregated output by anomaly_count desc then pct_breach desc and limit to top 20
os022_metric_anomaly_agg = os022_metric_anomaly_agg.sort_values(['anomaly_count', 'pct_breach'], ascending=[False, False]).head(20)

# Select and order columns as requested
os022_metric_anomaly_agg = os022_metric_anomaly_agg[[
    'cmdb_id', 'kpi_name', 'anomaly_type', 'earliest_anomaly_local',
    'anomaly_count', 'max_value', 'min_value', 'threshold_used', 'pct_breach'
]].reset_index(drop=True)

# Final display: aggregated result and raw rows (compact)
os022_metric_anomaly_agg, os022_metric_window_rows

```
Out[7]:
```
```python
# Summarize the os_022 metric anomaly aggregation and raw window rows (plain English).
try:
    agg = os022_metric_anomaly_agg  # aggregated anomalies (stored in kernel)
    rows = os022_metric_window_rows
except NameError:
    agg = None
    rows = None

if agg is None or agg.empty:
    summary = "No metric anomalies detected for os_022 in the specified incident window."
else:
    # There is at least one aggregated anomaly row
    # Build concise summary
    example = agg.iloc[0]
    cmdb = example['cmdb_id']
    kpi = example['kpi_name']
    atype = example['anomaly_type']
    earliest = example['earliest_anomaly_local']
    count = int(example['anomaly_count'])
    maxv = float(example['max_value'])
    minv = float(example['min_value'])
    thresh = float(example['threshold_used'])
    pct = float(example['pct_breach'])
    
    # Raw rows info
    if rows is None or rows.empty:
        raw_info = "No raw metric rows returned for os_022 in the window."
    else:
        n_raw = int(rows.shape[0])
        # count how many rows have missing thresholds
        missing_p95 = int(rows['threshold_p95'].isna().sum()) if 'threshold_p95' in rows.columns else n_raw
        missing_p5  = int(rows['threshold_p5'].isna().sum())  if 'threshold_p5' in rows.columns  else n_raw
        raw_info = (f"Returned {n_raw} raw metric rows for os_022 (sorted by timestamp). "
                    f"{missing_p95} rows lack threshold_p95 and {missing_p5} lack threshold_p5.")
    
    summary = (
        f"Incident window: 2020-05-26 02:30:00+08:00 through 2020-05-26 03:00:00+08:00.\n"
        f"Aggregated anomaly (top): {cmdb} | {kpi} | {atype}.\n"
        f"  - earliest_anomaly_local: {earliest}\n"
        f"  - anomaly_count: {count}\n"
        f"  - max_value: {maxv}, min_value: {minv}\n"
        f"  - threshold_used: {thresh}\n"
        f"  - pct_breach: {pct}\n\n"
        f"{raw_info}\n\n"
        f"Stored kernel variables: 'os022_metric_anomaly_agg' (aggregated) and 'os022_metric_window_rows' (raw rows)."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id                  kpi_name anomaly_type    earliest_anomaly_local anomaly_count  max_value  min_value  threshold_used  pct_breach
0  os_022  Outgoing_network_traffic          low  2020-05-26 02:42:00+0800             4   0.040945   0.019897        0.040945    0.514055,      timestamp           timestamp_local                  kpi_name       value  threshold_p95  threshold_p5
0   1590431400  2020-05-26 02:30:00+0800                  ss_total  320.000000            NaN           NaN
1   1590431400  2020-05-26 02:30:00+0800                Send_total    0.069894            NaN           NaN
2   1590431400  2020-05-26 02:30:00+0800             Shared_memory    0.000000            NaN           NaN
3   1590431400  2020-05-26 02:30:00+0800      Processor_load_5_min    0.060625            NaN           NaN
4   1590431400  2020-05-26 02:30:00+0800                   Page_pi    0.000000            NaN           NaN
..         ...                       ...                       ...         ...            ...           ...
45  1590431460  2020-05-26 02:31:00+0800                 ICMP_ping    1.000000            NaN           NaN
46  1590431460  2020-05-26 02:31:00+0800  Incoming_network_traffic    0.043826            NaN           NaN
47  1590431460  2020-05-26 02:31:00+0800                  ss_total  292.000000            NaN           NaN
48  1590431460  2020-05-26 02:31:00+0800      Processor_load_5_min    0.055938            NaN           NaN
49  1590431460  2020-05-26 02:31:00+0800                Recv_total   41.948341            NaN           NaN

[50 rows x 6 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```